# **02 컨텍스트 증강 및 프롬프트 엔지니어링**

### 학습 내용
1. 외부 정보(컨텍스트)를 프롬프트에 주입하기
2. 텍스트 파일을 활용한 정보 제공
3. 프롬프트 템플릿 활용
4. RAG의 기본 원리 이해

## 0. 환경 변수 설정

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("API Key가 설정되었습니다.")

API Key가 설정되었습니다.


## 1. LLM 초기화

In [2]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-5.4-mini")

## 2. 외부 정보 없이 질문하기

먼저 LLM이 알지 못하는 특정 정보에 대해 질문해봅시다.

In [3]:
from IPython.display import Markdown, display

# LLM이 모를 가능성이 높은 질문
question = "우리 회사의 여름 휴가 정책이 어떻게 되나요?"

response = llm.invoke(question)
display(Markdown(response.content))

회사의 **여름 휴가 정책**은 제가 현재 가지고 있는 정보만으로는 확인할 수 없습니다.  
보통은 아래 문서나 담당 부서에서 확인할 수 있어요:

- **사내 인사/복지 규정**
- **연차 및 휴가 안내 공지**
- **HR/인사팀 문의**
- **사내 포털 또는 메신저 공지**

원하시면 제가 대신 확인할 수 있도록,
1) 회사명이나  
2) 관련 공지/규정 문서 내용을 붙여주시면  
그 내용을 바탕으로 여름 휴가 정책을 정리해드릴게요.

LLM은 학습 데이터에 없는 특정 정보(회사 내부 정책, 개인 정보 등)는 답변할 수 없습니다.

이를 해결하기 위해 **외부 정보를 프롬프트에 포함**시킬 수 있습니다.

## 3. 컨텍스트를 직접 추가하여 질문하기

필요한 정보를 프롬프트에 직접 포함시켜 봅시다.

In [4]:
# 회사 정책 정보 (컨텍스트)
context = """
우리 회사 여름 휴가 정책:
- 전 직원은 7월~8월 중 연속 5일의 여름 휴가를 사용할 수 있습니다.
- 휴가 신청은 최소 2주 전에 해야 합니다.
- 부서별로 최소 인원이 유지되어야 하므로 팀장과 사전 협의가 필요합니다.
- 여름 휴가는 연차와 별도로 제공되는 특별 휴가입니다.
"""

# 컨텍스트와 질문을 함께 전달
prompt = f"""
다음 정보를 참고하여 질문에 답변하세요.

[참고 정보]
{context}

[질문]
{question}
"""

response = llm.invoke(prompt)
display(Markdown(response.content))

우리 회사의 여름 휴가 정책은 다음과 같습니다.

- **대상:** 전 직원
- **사용 기간:** **7월~8월 중**
- **휴가 일수:** **연속 5일**
- **신청 시기:** **최소 2주 전**에 신청해야 함
- **사전 협의:** 부서별 최소 인원 유지를 위해 **팀장과 사전 협의 필요**
- **휴가 성격:** **연차와 별도로 제공되는 특별 휴가**

원하시면 제가 이 내용을 **사내 공지문 형태**로도 정리해드릴게요.

## 4. 텍스트 파일로부터 정보 읽어오기

실제 상황에서는 정보가 파일, 데이터베이스, 웹 페이지 등에 저장되어 있습니다.

텍스트 파일에서 정보를 읽어와서 프롬프트에 주입해봅시다.

In [5]:
# 먼저 샘플 텍스트 파일을 생성합니다
sample_text = """
상명대학교 AI 서비스 개발 과정 안내

과정명: RAG · AI Agent 기반 실무형 AI 서비스 개발 과정
기간: 2026.08.18 ~ 2026.08.31 (10일, 총 80시간)
장소: 상명대학교 천안캠퍼스

주요 학습 내용:
1. LLM 애플리케이션 개발 기초
2. RAG(검색증강생성) 시스템 구축
3. Text2SQL 기반 데이터 조회 자동화
4. AI Agent 시스템 개발
5. LangGraph를 활용한 워크플로우 구성

최종 포트폴리오:
- RAG · Text2SQL 기반 데이터 조회 시스템
- Tool 기반 AI Agent 시스템
"""

# 파일 저장
with open("course_info.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)

print("샘플 텍스트 파일이 생성되었습니다: course_info.txt")

샘플 텍스트 파일이 생성되었습니다: course_info.txt


In [6]:
# 텍스트 파일 읽기
with open("course_info.txt", "r", encoding="utf-8") as f:
    course_context = f.read()

print("파일 내용:")
print(course_context)

파일 내용:

상명대학교 AI 서비스 개발 과정 안내

과정명: RAG · AI Agent 기반 실무형 AI 서비스 개발 과정
기간: 2026.08.18 ~ 2026.08.31 (10일, 총 80시간)
장소: 상명대학교 천안캠퍼스

주요 학습 내용:
1. LLM 애플리케이션 개발 기초
2. RAG(검색증강생성) 시스템 구축
3. Text2SQL 기반 데이터 조회 자동화
4. AI Agent 시스템 개발
5. LangGraph를 활용한 워크플로우 구성

최종 포트폴리오:
- RAG · Text2SQL 기반 데이터 조회 시스템
- Tool 기반 AI Agent 시스템



In [7]:
# 파일에서 읽은 정보를 활용하여 질문하기
question = "이 과정의 학습 기간과 주요 학습 내용을 요약해주세요."

prompt = f"""
다음 과정 안내 정보를 참고하여 질문에 답변하세요.

[과정 정보]
{course_context}

[질문]
{question}
"""

response = llm.invoke(prompt)
display(Markdown(response.content))

이 과정은 **2026.08.18부터 2026.08.31까지 10일간, 총 80시간** 진행됩니다.  
주요 학습 내용은 다음과 같습니다.

- **LLM 애플리케이션 개발 기초**
- **RAG(검색증강생성) 시스템 구축**
- **Text2SQL 기반 데이터 조회 자동화**
- **AI Agent 시스템 개발**
- **LangGraph를 활용한 워크플로우 구성**

즉, **LLM 기반 서비스 개발의 기초부터 RAG, Text2SQL, AI Agent, LangGraph 워크플로우까지 실무 중심으로 학습하는 과정**입니다.

## 5. 프롬프트 템플릿 활용하기

LangChain의 `PromptTemplate`을 사용하면 프롬프트를 더 체계적으로 관리할 수 있습니다.

In [8]:
from langchain_core.prompts import PromptTemplate

# 프롬프트 템플릿 정의
template = """
당신은 도움이 되는 AI 어시스턴트입니다.
주어진 정보를 바탕으로 사용자의 질문에 정확하고 친절하게 답변하세요.

[참고 정보]
{context}

[질문]
{question}

[답변]
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

# 템플릿에 값 채우기
formatted_prompt = prompt_template.format(
    context=course_context,
    question="이 과정에서 어떤 포트폴리오를 완성하나요?"
)

print("생성된 프롬프트:")
print(formatted_prompt)
print("\n" + "="*80 + "\n")

response = llm.invoke(formatted_prompt)
print("답변:")
display(Markdown(response.content))

생성된 프롬프트:

당신은 도움이 되는 AI 어시스턴트입니다.
주어진 정보를 바탕으로 사용자의 질문에 정확하고 친절하게 답변하세요.

[참고 정보]

상명대학교 AI 서비스 개발 과정 안내

과정명: RAG · AI Agent 기반 실무형 AI 서비스 개발 과정
기간: 2026.08.18 ~ 2026.08.31 (10일, 총 80시간)
장소: 상명대학교 천안캠퍼스

주요 학습 내용:
1. LLM 애플리케이션 개발 기초
2. RAG(검색증강생성) 시스템 구축
3. Text2SQL 기반 데이터 조회 자동화
4. AI Agent 시스템 개발
5. LangGraph를 활용한 워크플로우 구성

최종 포트폴리오:
- RAG · Text2SQL 기반 데이터 조회 시스템
- Tool 기반 AI Agent 시스템


[질문]
이 과정에서 어떤 포트폴리오를 완성하나요?

[답변]



답변:


이 과정의 최종 포트폴리오는 **2가지**입니다.

1. **RAG · Text2SQL 기반 데이터 조회 시스템**
2. **Tool 기반 AI Agent 시스템**

즉, 검색증강생성(RAG)과 Text2SQL을 활용한 데이터 조회 시스템, 그리고 도구를 활용하는 AI Agent 시스템을 완성하게 됩니다.

## 6. RAG의 기본 원리 이해

지금까지 실습한 내용이 바로 **RAG(Retrieval-Augmented Generation)** 의 핵심 원리입니다.

### RAG의 기본 흐름

1. **사용자 질문 입력**
2. **관련 문서 검색** (Retrieval)
   - 벡터 데이터베이스, 키워드 검색, 하이브리드 검색 등
3. **검색된 문서를 프롬프트에 주입** (Augmentation)
4. **LLM이 컨텍스트를 바탕으로 답변 생성** (Generation)

현재까지는 문서 검색 없이 직접 컨텍스트를 제공했지만,
다음 실습에서는 **벡터 데이터베이스를 활용한 자동 문서 검색**을 구현합니다.

## 📖 과제 1: 나만의 지식 베이스 만들기

자신이 관심 있는 주제나 전공 분야의 정보를 담은 텍스트 파일을 만들고,
해당 정보를 활용하여 질문-답변 시스템을 구현해보세요.

**예시 주제:**
- 좋아하는 영화/드라마의 줄거리와 등장인물 정보
- 자신의 포트폴리오나 이력서 내용
- 관심 분야의 용어 사전
- 수업 노트나 요약 자료

**구현 요구사항:**
1. 텍스트 파일(.txt) 생성 (최소 200자 이상)
2. 파일 내용을 읽어와서 컨텍스트로 활용
3. 3개 이상의 질문을 만들어 답변 생성

In [ ]:
# TODO 1. 나만의 지식 베이스 내용 작성 (200자 이상)
my_knowledge_base = """
[여기에 자신이 관심 있는 주제의 내용을 작성하세요]

예시 주제:
- 좋아하는 영화/드라마의 줄거리와 등장인물 정보
- 자신의 포트폴리오나 이력서 내용
- 관심 분야의 용어 사전
- 수업 노트나 요약 자료

[이 부분을 지우고 실제 내용으로 채워주세요]
"""

# TODO 2. 파일명 설정 (예: "my_movie_info.txt", "my_portfolio.txt")
filename = "my_knowledge.txt"

# TODO 3. 질문 3개 작성
questions = [
    "질문 1을 여기에 작성하세요",
    "질문 2를 여기에 작성하세요",
    "질문 3을 여기에 작성하세요"
]

# 텍스트 파일 생성
with open(filename, "w", encoding="utf-8") as f:
    f.write(my_knowledge_base)
print(f"✓ 파일이 생성되었습니다: {filename}\n")

# 파일 내용 읽기
with open(filename, "r", encoding="utf-8") as f:
    context = f.read()

# 프롬프트 템플릿 정의
from langchain_core.prompts import PromptTemplate

template = """
당신은 도움이 되는 AI 어시스턴트입니다.
주어진 정보를 바탕으로 사용자의 질문에 정확하고 친절하게 답변하세요.

[참고 정보]
{context}

[질문]
{question}

[답변]
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

# 각 질문에 대해 답변 생성
from IPython.display import Markdown, display

for i, question in enumerate(questions, 1):
    print(f"\n{'='*80}")
    print(f"질문 {i}: {question}")
    print('='*80)

    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    print("생성된 프롬프트:")
    print(formatted_prompt)
    print("\n" + "="*80 + "\n")

    response = llm.invoke(formatted_prompt)
    print("답변:")
    display(Markdown(response.content))

## 📖 과제 2: 프롬프트 최적화하기

같은 컨텍스트와 질문이라도 프롬프트를 어떻게 구성하느냐에 따라 답변 품질이 달라집니다.

다음 요소들을 추가하여 프롬프트를 개선해보세요:

1. **역할 정의**: "당신은 ~한 전문가입니다"
2. **답변 형식 지정**: "다음 형식으로 답변하세요: ..."
3. **제약 조건**: "정보에 없는 내용은 '정보 없음'이라고 답하세요"
4. **예시 제공**: Few-shot learning (예시 포함)

원본 프롬프트와 개선된 프롬프트의 답변을 비교해보세요.

---

### 참고 자료

- [LangChain Prompts 공식 문서](https://python.langchain.com/docs/modules/model_io/prompts/)
- [프롬프트 엔지니어링 가이드](https://www.promptingguide.ai/)